In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
print(df["labels"].unique())
print(df["labels"].value_counts(dropna=False))
print(df["labels"].dtype)

[0 1 2]
labels
2    72250
1    55213
0    35510
Name: count, dtype: int64
int64


Now that your model is trained and saved, you can load it and use it for inference on new text. Here's how you can do that:

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import os
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
驱动 = "cuda" if torch.cuda.is_available() else "cpu"

# 1. DATA PREPROCESSING
df = pd.read_csv("Twitter_Data.csv")
df = df.rename(columns={"clean_text": "text", "category": "labels"})

# Fill missing text and ensure labels are valid integers
df["text"] = df["text"].fillna("neutral").astype(str)
df["labels"] = pd.to_numeric(df["labels"], errors="coerce")
df = df.dropna(subset=["labels"])

# Map labels from [-1, 0, 1] to [0, 1, 2]
df["labels"] = df["labels"].replace({-1: 0, 0: 1, 1: 2}).astype(int)

# CRITICAL FIX: Ensure no labels exist outside [0, 1, 2]
df = df[df["labels"].isin([0, 1, 2])].reset_index(drop=True)

print(f"Dataset size: {len(df)}")
print(f"Verified Label classes: {df['labels'].unique()}")

# 2. DATASET PREPARATION
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_checkpoint = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 3. MODEL SETUP
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

# 4. TRAINING ARGUMENTS
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="sentiment_roberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32, # Increased for speed
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available() # Uses mixed precision if on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# 5. EXECUTION
trainer.train()

# 6. EVALUATION & REPORTING
print("\n--- Final Evaluation ---")
results = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(results.predictions, axis=-1)
actual = results.label_ids

print("\nClassification Report:")
print(classification_report(actual, preds, target_names=["negative", "neutral", "positive"]))

# 7. SAVE
model.save_pretrained("sentiment_roberta_model")
tokenizer.save_pretrained("sentiment_roberta_tokenizer")
print("Model and Tokenizer saved successfully.")

Dataset size: 162973
Verified Label classes: [0 1 2]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/130378 [00:00<?, ? examples/s]

Map:   0%|          | 0/32595 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


--- Final Evaluation ---



Classification Report:
              precision    recall  f1-score   support

    negative       0.95      0.96      0.95      7200
     neutral       0.98      0.98      0.98     10916
    positive       0.98      0.98      0.98     14479

    accuracy                           0.97     32595
   macro avg       0.97      0.97      0.97     32595
weighted avg       0.97      0.97      0.97     32595



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and Tokenizer saved successfully.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

model_path = "sentiment_roberta_model"
tokenizer_path = "sentiment_roberta_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

label_map = {0: "negative", 1: "neutral", 2: "positive"}

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        logits = model(**inputs).logits

    pred = torch.argmax(logits).item()
    return label_map[pred]

# TEST YOUR MODEL
print(predict("It is fine."))
print(predict("The update was released."))
print(predict("Weather is normal today."))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

positive
neutral
positive


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("sentiment_model", 'zip', "sentiment_roberta_model")
shutil.make_archive("sentiment_tokenizer", 'zip', "sentiment_roberta_tokenizer")

files.download("sentiment_model.zip")
files.download("sentiment_tokenizer.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>